# 17. Ders: Yapay Zekada Veri Kirliliği ve Toksisite (Watchdog - Bölüm 1)

## Adım 1: Teorik Çerçeve ve Günlük Hayat Analojisi (Tay Botu Faciası)

2016 yılında Microsoft, Twitter (X) üzerinde kendi kendine öğrenen 'Tay' adında bir sohbet botu yayınladı. Tay, insanlarla konuştukça onların yazdıklarından öğreniyordu. İnternet trolleri Tay'a saatler boyunca ırkçı ve küfürlü şeyler yazdı. Sonuç? Sadece 16 saat içinde, masum yapay zeka Tay küfürbaz, ırkçı bir bota dönüştü ve fişi çekildi.

Yapay zeka ahlakı veya kuralları doğuştan bilmez, sadece ona verilen **veriyi (Data)** yansıtır. Eğer eğittiğiniz veriler internetin karanlık köşelerinden (küfür, nefret söylemi) geliyorsa, modeliniz de 'Toksik' (Zehirli) olacaktır. GCA Coding Jam 'Watchdog' (Veri Bekçisi) projemizde, internetteki toksik yazıları (küfürleri, nefret söylemlerini) tespit edip platformumuzu temiz tutacak bir yapay zeka bekçisi geliştireceğiz.

## Adım 2: Teknik Terimler ve Sözlük

- Data Bias (Veri Yanlılığı/Kirliliği): Modele verilen eğitim verilerinin tek taraflı, ayrımcı veya zehirli metinlerden oluşması sebebiyle modelin yanlış öğrenmesi durumu.
- Toxicity (Toksisite): Bir metnin (veya kişinin) diğer insanları rahatsız edici, saldırgan veya küfürlü olma derecesi.
- Hate Speech (Nefret Söylemi): Irk, din, cinsiyet gibi konularda toplulukları hedef alan zararlı metinler.
- Toxicity Classification (Toksisite Sınıflandırması): Bir metni 0 ile 1 arasında (0: çok temiz, 1: çok küfürlü/zararlı) değerlendiren makine öğrenmesi sistemleri.

## Adım 3: Hugging Face Toksisite Modelini Koda Aktarma

İnternet yorumlarını denetlemek (moderasyon) için 'text-classification' veya özel toksisite modelleri kullanılır. Hugging Face'te bulunan `unitary/toxic-bert` modeli bu iş için eğitilmiş dünyaca ünlü bir siber güvenlik modelidir.

In [ ]:
from transformers import pipeline

# Text-Classification görevi ile özel Toxic-BERT modelimizi indiriyoruz.
# (Bu model İngilizce küfür ve toksik kelimeleri tanır)
bekci_model = pipeline("text-classification", model="unitary/toxic-bert")

print("Watchdog (Veri Bekçisi) göreve başladı.")

## Adım 4: Zararsız ve Zararlı Cümleleri Test Etme

Bekçimize sıradan bir mesaj ile oyunlarda sıkça görülen kışkırtıcı (toksik) bir mesaj verelim ve farkı görelim.

In [ ]:
temiz_mesaj = "Good game everyone, it was fun playing with you."
toksik_mesaj = "You are all stupid losers, delete the game!"

sonuc_temiz = bekci_model(temiz_mesaj)
sonuc_toksik = bekci_model(toksik_mesaj)

print("Temiz Mesaj Analizi:", sonuc_temiz)
print("Toksik Mesaj Analizi:", sonuc_toksik)

# Çıktılarda 'toxic' etiketi ve bu etiketin 'score' (eminlik) değerini inceleyin.
# Temiz mesajda toksik olma ihtimali çok düşük (Örn: 0.0005) çıkarken, toksik mesajda %95'lere (0.95) çıkacaktır.

## Adım 5: İnternet Yorumlarını Simüle Eden Bir DataFrame

Sosyal medyadaki (örneğin YouTube veya Instagram) yorum akışını Pandas kullanarak simüle edelim. Watchdog bu yorumları tarayacak.

In [ ]:
import pandas as pd

sosyal_medya_yorumlari = {
    "Kullanici": ["Ali123", "ProGamer", "Ayse_99", "TrollKing"],
    "Yorum": [
        "This video is very helpful, thank you!",
        "What a terrible video, you are an idiot.",
        "Can you make a video about Python next?",
        "I hate you and your channel, shut up!"
    ]
}

df_yorumlar = pd.DataFrame(sosyal_medya_yorumlari)
print("--- GELEN YORUMLAR (FİLTRESİZ) ---")
print(df_yorumlar)

## Adım 6: Döngü ile Tüm Yorumları Bekçiden Geçirme

Veri çerçevemizdeki her bir yorumu teker teker modele gönderip toksik (zararlı) olup olmadığını bulacağız ve toksiklik derecesini (score) yüzde olarak yazdıracağız.

In [ ]:
print("--- WATCHDOG DENETİM RAPORU ---\n")

for index, row in df_yorumlar.iterrows():
    kisi = row["Kullanici"]
    mesaj = row["Yorum"]
    
    # Modeli çalıştır
    analiz = bekci_model(mesaj)[0]
    skor = analiz['score']
    
    print(f"Kullanıcı: {kisi}")
    print(f"Yorum: {mesaj}")
    print(f"Zararlılık Skoru: %{skor * 100:.2f}\n")

## Adım 7: Güvenlik Sınırı (Threshold) Belirleme

Skor %0.05 de olabilir, %99 da olabilir. Sistem nerede bu yorumu silmelidir? Genellikle %60 (0.60) eşiği güvenlik sistemlerinde başlangıç kabul edilir.

In [ ]:
GUVENLIK_SINIRI = 0.60

test_mesaji = "You are so dumb."
skor = bekci_model(test_mesaji)[0]['score']

if skor > GUVENLIK_SINIRI:
    print(f"SİSTEM UYARISI: Bu mesaj (%{skor*100:.1f} toksik) güvenlik sınırını aştı ve silindi!")
else:
    print("Mesaj güvenli, yayınlanmasına izin verildi.")

## Adım 8: Dene ve Keşfet - Modelin Kafasını Karıştırmak

Bazen insanlar arkadaşlarına şaka yollu hakaret ederler (Örn: "You are crazy man, I love this!"). Model bu cümleyi "crazy" kelimesinden dolayı zararlı mı sayacak, yoksa "love" kelimesinden dolayı şaka olduğunu mu anlayacak?

Aşağıdaki kodda `saka_mesaji` değişkenini çalıştırın ve yapay zekanın bunu (Threshold'u) aşıp aşmadığını test edin.

In [ ]:
saka_mesaji = "You are crazy man, I love this!"

saka_skoru = bekci_model(saka_mesaji)[0]['score']
print(f"Şaka cümlesinin Toksik Skoru: %{saka_skoru*100:.2f}")

# Sonuca göre modelin bağlamı anlama gücünü tartışın.

## Adım 9: Sıra Sende Görevi - Küfür Sansürleme Fonksiyonu

Problem: Oyun şirketleri yorumları tamamen silmek yerine bazen küfürleri yıldızlarla (***) sansürleyerek yayınlarlar. Basit bir sansür fonksiyonu yazmanız isteniyor.

Gereksinimler:
1. `sansurle(mesaj)` adında bir fonksiyon yazın.
2. Fonksiyon gelen mesajı `bekci_model` ile analiz etsin.
3. Eğer toksik skoru `0.50`'den büyükse, mesajın İÇERİĞİ yerine ekrana şu uyarıyı döndürsün (return): `"[Bu mesaj topluluk kuralları gereği gizlenmiştir]"`
4. Eğer toksik skoru `0.50`'den küçükse (zararsızsa), mesajın orjinalini olduğu gibi geri döndürsün.
5. Dışarıda fonksiyonu bir kötü ("You are an idiot") bir de iyi ("Hello my friend") mesajla çağırıp `print()` ile test edin.

In [ ]:
# GÖREV KODLARINIZI BURAYA YAZINIZ:



Tebrikler! 17. Dersi tamamlayarak yapay zekanın sadece üretici değil, aynı zamanda toplumları koruyucu bir 'Denetmen' (Watchdog) olarak da kullanılabileceğini öğrendiniz.

Bir sonraki derste, bu altyapıyı geliştirerek Orta Risk, Yüksek Risk gibi kademeli moderasyonlar ve Yanlış Pozitif (False Positive) sorunlarını ele alacağımız 2. bölüme geçeceğiz.